## Feature Enginnering

Esse notebook cria a tabela base para o treinamento.

In [0]:
%pip install -r ../requirements.txt

In [0]:
import pandas as pd

In [0]:
import sys, os

sys.path.append(os.path.abspath("../src"))#Sobe um diretório

%load_ext autoreload
%autoreload 2

from feature_utils import MiceImputerTransformer,OutlierIndicatorTransformer,CappingTransformer,FeatureCyclic,FeatureDateExtractor,adicionar_periodo_dia# Importa da pasta src

In [0]:
X = pd.read_csv('/Volumes/catalog_samarco/storage_datalake/dbw-otimizacao/pluma_multiclasse_5krows/df_reduzida.csv')
X = (

    adicionar_periodo_dia(X, coluna_timestamp_str='t')

    .rename(columns={'periodo': 'Dia/Noite',"Unnamed: 0":'id'})

    .loc[lambda x: x['Dia/Noite'] == 'Dia']

    .assign(

        Tempo=lambda x: (pd.to_timedelta(x['Hora.1']).dt.total_seconds() / 60).astype(int)

    )

    .drop(columns=["id","Amostragem","Dia/Noite", "t", "n", "Hora.1",'Ano','Data','Nascer do sol','Pôr do sol','Dia?','Y','Legenda Y'])
    .pipe(pd.get_dummies,columns=['Campanha'],dtype=int)

)

X

In [0]:
colunas_inputar = [colunas for colunas in X.columns if '[US3]' in colunas or '[US4]' in colunas]
colunas_inputar

In [0]:
import pandas as pd
import re
from databricks.feature_store import FeatureStoreClient


def clean_column_names(df):
    """Substitui caracteres especiais e espaços por underlines."""
    new_columns = []
    for col in df.columns:
        # Substitui espaços e símbolos ((), [], °, etc) por '_'
        clean_col = re.sub(r'[^\w]', '_', col) 
        # Remove underlines duplicados (ex: ___)
        clean_col = re.sub(r'_+', '_', clean_col)
        # Remove underline do final ou começo
        clean_col = clean_col.strip('_')
        new_columns.append(clean_col)
    
    df.columns = new_columns
    return df

mice = MiceImputerTransformer(
    cols_to_impute=colunas_inputar, 
    max_iter=20,
    random_state=42
)

df_pandas = mice.fit_transform(df)

df_pandas = df_pandas.reset_index(names=['id_sequencial']) 

df_pandas['id_sequencial'] = df_pandas['id_sequencial'].astype(str)


df_pandas = clean_column_names(df_pandas)

print("Exemplo de colunas limpas:", df_pandas.columns.tolist()[:5])

# --- 5. Salvando ---
df_spark = spark.createDataFrame(df_pandas)
fs = FeatureStoreClient()
table_name = "catalog_samarco.gea_sandbox.pluma_features_gold"

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

fs.create_table(
    name=table_name,
    primary_keys=["id_sequencial"], # Usamos o nome que demos no reset_index
    df=df_spark,
    description="Features Gold Pluma - Nomes higienizados"
)

print(f"✅ Sucesso! Tabela salva em: {table_name}")

In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()
table_name = "catalog_samarco.gea_sandbox.pluma_features_gold"


X = fs.read_table(name=table_name)

display(X)